# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Storage Solutions (PostgreSQL)** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [1]:
from pcamarillor.spark_utils import SparkUtils

su = SparkUtils("Examples on storage solutions with PosgreSQL",
                "spark://spark-master:7077",
                spark_jars="/opt/spark/work-dir/notebooks/jars/postgresql-42.7.8.jar ")
su.spark

26/03/11 22:05:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


# Create DataFrames

In [2]:
import pyspark.sql.functions as F
movies_columns = [("MovieID", "int"),
                  ("Title", "string"),
                  ("Genre", "string"),
                  ("ReleaseYear", "int"),
                  ("ReleaseDate_Str", "string"),
                  ("Country", "string"),
                  ("BudgetUSD", "float"),
                  ("US_BoxOfficeUSD", "float"),
                  ("Global_BoxOfficeUSD", "float"),
                  ("Opening_Day_SalesUSD", "float"),
                  ("One_Week_SalesUSD", "float"),
                  ("IMDbRating", "float"),
                  ("RottenTomatoesScore", "float"),
                  ("NumVotesIMDb", "int"),
                  ("NumVotesRT", "int"),
                  ("Director", "string"),
                  ("LeadActor", "string")]

movies_schema = SparkUtils.generate_schema(movies_columns)

movies_date_str_df = (su.spark.read 
                .option("header", "true")
                .schema(movies_schema)
                .csv("hdfs://namenode:9000/movies/"))

movies_df = movies_date_str_df.withColumn("ReleaseDate", F.to_date("ReleaseDate_Str", "dd-MM-yyyy")).drop("ReleaseDate_Str")
movies_df = movies_df.groupBy("Genre").count().orderBy(F.col("count"), ascending=False)
movies_df.show()

+-----------+------+
|      Genre| count|
+-----------+------+
|      Drama|250018|
|     Comedy|199832|
|     Action|150131|
|   Thriller|100071|
|    Romance|100021|
|     Horror|100010|
|Documentary| 50114|
|     Sci-Fi| 49802|
+-----------+------+



# Write data to a PostgreSQL DB

## Install PostgreSQL with Docker


    docker run -d --name postgres-iteso --network spark-cluster_default -e POSTGRES_PASSWORD=Admin@1234 postgres

In [ ]:
jdbc_url = "jdbc:postgresql://postgres-iteso:5432/postgres"
table_name = "airlines_transformed"

movies_df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", table_name) \
    .option("user", "postgres") \
    .option("password", "Admin@1234") \
    .option("driver", "org.postgresql.Driver") \
    .save()

print("DataFrame successfully written into a PosgreSQL DB !")

DataFrame successfully written into a PosgreSQL DB !


# Read data from a PosgreSQL DB

In [5]:
jdbc_url = "jdbc:postgresql://postgres-iteso:5432/postgres"
db_properties = {
      "user": "postgres",      
      "password": "Admin@1234",
      "driver": "org.postgresql.Driver"
  }

df = (su.spark.read
    .jdbc(url=jdbc_url, table=table_name, properties=db_properties))

df.printSchema()
df.show(5, truncate=False)

root
 |-- Genre: string (nullable = true)
 |-- count: long (nullable = true)

+--------+------+
|Genre   |count |
+--------+------+
|Drama   |250018|
|Comedy  |199832|
|Action  |150131|
|Thriller|100071|
|Romance |100021|
+--------+------+
only showing top 5 rows


In [6]:
su.spark.stop()